# Huấn luyện mô hình Titanic (Training Models)

## Import thư viện

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import joblib  # Để lưu model

## 1. Mục tiêu huấn luyện
+ **Mô tả**:
    + Chia train/val từ processed data (sử dụng dữ liệu tối ưu: train_final.csv).
    + Thử nhiều model: Logistic, RF, XGBoost, SVC, KNN.
    + Đánh giá bằng Accuracy, F1, ROC-AUC.
    + Chọn best model (dựa trên F1/ROC vì imbalance).
+ **Dữ liệu vào**: Từ processed/train_final.csv.
+ **Kết quả**: Model tốt nhất lưu vào saved_models.

## 2. Load dữ liệu từ processed

In [10]:
# Load train processed (chỉ train có Survived)
# *** Đã thay đổi đường dẫn để sử dụng dữ liệu TỐI ƯU ***
train_path = '../data/processed/train_processed_v3.csv' 
df_train = pd.read_csv(train_path)

X = df_train.drop('Survived', axis=1)
y = df_train['Survived']

# Split train/val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Dữ liệu train_final.csv đã được tải và chia thành Train/Validation.")
print(f"Kích thước tập Train: {X_train.shape}, Validation: {X_val.shape}")

Dữ liệu train_final.csv đã được tải và chia thành Train/Validation.
Kích thước tập Train: (712, 19), Validation: (179, 19)


## 3. Huấn luyện và đánh giá models
+ Thử nhiều model và in metrics.

In [11]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'SVC': SVC(kernel='rbf', probability=True, random_state=42, C=1.0, gamma='scale'),
    'K-Neighbours': KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan', p=1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_prob) if y_prob is not None else None

    print(f"{name} Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {f'{roc_auc:.4f}' if roc_auc is not None else 'N/A'}")
    print("-" * 30)

    results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'ROC_AUC': roc_auc})

# Lưu metrics
pd.DataFrame(results).to_csv('../models/metrics/evaluation_results.csv', index=False)

Logistic Regression Results:
  Accuracy: 0.8436
  F1 Score: 0.7879
  ROC AUC:  0.8687
------------------------------
Random Forest Results:
  Accuracy: 0.7877
  F1 Score: 0.7246
  ROC AUC:  0.8292
------------------------------
XGBoost Results:
  Accuracy: 0.7989
  F1 Score: 0.7353
  ROC AUC:  0.8382
------------------------------
SVC Results:
  Accuracy: 0.8268
  F1 Score: 0.7634
  ROC AUC:  0.8691
------------------------------
K-Neighbours Results:
  Accuracy: 0.7709
  F1 Score: 0.7007
  ROC AUC:  0.7926
------------------------------


## 4. Chọn và lưu best model
+ Chọn best model (dựa trên metrics ở bước 3).
+ Train full train data.

In [12]:
# Tùy theo kết quả ở bước 3, chọn model tốt nhất 
best_model = LogisticRegression(random_state=42, max_iter=1000)
best_model.fit(X, y)  # Train full train data (X, y là full train set)

# Lưu model
joblib.dump(best_model, '../models/saved_models/best_svc_model.pkl')
print("Best model saved to best_svc_model.pkl.")

Best model saved to best_svc_model.pkl.


# Kết thúc